In [ ]:
import pandas as pd
import requests

BASE = "https://www.fema.gov/api/open/v3"

def get(entity, **params):
    r = requests.get(f"{BASE}/{entity}", params=params, timeout=60)
    r.raise_for_status()
    return r.json()

In [37]:
resp = get("NfipClaims", **{"$top": 2})
print(resp.keys())
print(resp["metadata"])

dict_keys(['metadata', 'NfipClaims'])
{'skip': 0, 'select': None, 'rundate': '2026-11-18T14:43:34.516Z', 'top': 2, 'filter': '', 'format': 'json', 'metadata': True, 'orderby': '', 'entityname': 'NfipClaims', 'version': 'v3', 'url': '/api/open/v3/NfipClaims?%24top=2', 'count': 0}


In [38]:
resp = get("NfipClaims", **{"$top": 1, "$filter": "state eq 'NC'", "$count": "true"})
print(resp["metadata"]["count"])

109642


In [39]:
cols = ("id,dateOfLoss,yearOfLoss,state,countyCode,censusGeoid,ratedFloodZone,"
        "occupancyType,elevatedBuildingIndicator,causeOfDamage,floodEvent,"
        "amountPaidOnBuildingClaim,amountPaidOnContentsClaim,buildingDamageAmount,"
        "totalBuildingInsuranceCoverage,reportedCity,asOfDate")

resp = get("NfipClaims", **{
    "$top": resp["metadata"]["count"],
    "$filter": "state eq 'NC' and yearOfLoss ge 2019",
    "$select": cols,
    "$orderby": "id",
    "$count": "true",
})

claims = pd.DataFrame(resp["NfipClaims"])
print(len(claims))
claims.head()

8984


,id,dateOfLoss,yearOfLoss,state,countyCode,censusGeoid,ratedFloodZone,occupancyType,elevatedBuildingIndicator,causeOfDamage,floodEvent,amountPaidOnBuildingClaim,amountPaidOnContentsClaim,buildingDamageAmount,totalBuildingInsuranceCoverage,reportedCity,asOfDate
0,11604498,2019-01-22T00:00:00.000Z,2019,NC,37149,371499202003,X,1,False,NaN,NaN,NaN,NaN,NaN,250000,Currently Unavailable,2026-11-16T00:00:00.000Z
1,11604624,2019-01-11T00:00:00.000Z,2019,NC,37051,370510032052,X,1,True,4,NaN,998.40,0.0,1997.0,250000,Currently Unavailable,2026-11-16T00:00:00.000Z
2,11604677,2019-01-22T00:00:00.000Z,2019,NC,37019,370190205102,VE,6,False,0,NaN,NaN,NaN,NaN,500000,Currently Unavailable,2026-11-16T00:00:00.000Z
3,11604754,2019-01-05T00:00:00.000Z,2019,NC,37129,371290122013,AE,1,True,0,NaN,NaN,NaN,NaN,221400,Currently Unavailable,2026-11-16T00:00:00.000Z
4,11604767,2019-01-01T00:00:00.000Z,2019,NC,37053,370531101012,AE,1,True,0,NaN,2120.62,0.0,3371.0,235600,Currently Unavailable,2026-11-16T00:00:00.000Z


In [40]:
print(len(claims), resp["metadata"]["count"])

8984 8984


In [41]:
print(claims.dtypes)
print(claims.isna().mean().sort_values(ascending=False).head(10))
print(claims["ratedFloodZone"].value_counts().head(20))
print(claims["occupancyType"].value_counts())
print(claims["floodEvent"].value_counts().head(10))
print(claims["reportedCity"].value_counts().head(10))
claims[["amountPaidOnBuildingClaim", "buildingDamageAmount"]].describe()


id                                  int64
dateOfLoss                            str
yearOfLoss                          int64
state                                 str
countyCode                            str
censusGeoid                           str
ratedFloodZone                        str
occupancyType                       int64
elevatedBuildingIndicator            bool
causeOfDamage                         str
floodEvent                            str
amountPaidOnBuildingClaim         float64
amountPaidOnContentsClaim         float64
buildingDamageAmount              float64
totalBuildingInsuranceCoverage      int64
reportedCity                          str
asOfDate                              str
dtype: object
floodEvent                   0.409728
buildingDamageAmount         0.108749
amountPaidOnBuildingClaim    0.073687
amountPaidOnContentsClaim    0.073687
causeOfDamage                0.016140
censusGeoid                  0.003451
ratedFloodZone               0.001113
county

,amountPaidOnBuildingClaim,buildingDamageAmount
count,8322.000000,8.007000e+03
mean,43316.277098,5.586420e+04
std,80545.586277,5.987218e+05
min,-6251.000000,0.000000e+00
25%,2942.997500,4.734000e+03
50%,11840.605000,1.368300e+04
75%,42527.297500,4.552900e+04
max,515868.360000,5.284099e+07


In [42]:
resp = get("NfipPolicies", **{
    "$top": 1, # this is just to get the count, so we don't need more than 1 row
    "$filter": "propertyState eq 'NC' and policyEffectiveDate ge '2019-01-01T00:00:00.000Z'", # this is the same filter as the claims query, but for policies
    "$count": "true", # this is the key to getting the count of policies that match the filter
})

print(resp["metadata"]["count"])
pd.DataFrame(resp["NfipPolicies"]).T

962804


,0
agricultureStructureIndicator,False
asOfDate,2026-11-11T00:00:00.000Z
baseFloodElevation,3250.5
basementEnclosureCrawlspaceType,None
cancellationDateOfFloodPolicy,None
...,...
fullRiskPremium,2080
buildingOverWaterType,1
foundationType,4
preFIRMConstructionIndicator,False


In [43]:
c = set(get("NfipClaims", **{"$top": 1})["NfipClaims"][0])
p = set(get("NfipPolicies", **{"$top": 1})["NfipPolicies"][0])
print("Only i n claims:", sorted(c - p))
print("Only in policies:", sorted(p - c))

Only i n claims: ['amountPaidOnBuildingClaim', 'amountPaidOnContentsClaim', 'amountPaidOnIncreasedCostOfComplianceClaim', 'buildingDamageAmount', 'buildingPropertyValue', 'causeOfDamage', 'contentsDamageAmount', 'contentsPropertyValue', 'contentsReplacementCost', 'countyCode', 'dateOfLoss', 'disasterAssistanceCoverageRequired', 'eventDesignationNumber', 'exteriorWaterDepth', 'ficoNumber', 'floodCharacteristicsIndicator', 'floodEvent', 'floodWaterDuration', 'houseWorship', 'iccCoverage', 'interiorWaterDepth', 'mostRecentPaymentDate', 'mostRecentRecoveryDate', 'netBuildingPaymentAmount', 'netContentsPaymentAmount', 'netIccPaymentAmount', 'nonPaymentReasonBuilding', 'nonPaymentReasonContents', 'numberOfFloorsInTheInsuredBuilding', 'numberOfUnits', 'openDate', 'preFirmIndicator', 'replacementCostBasis', 'state', 'totalBldgClaimPmtRecovery', 'totalContentsClaimPmtRecovery', 'totalIccClaimPmtRecovery', 'totalSalvageRecovery', 'totalSubrogationRecovery', 'waterDepth', 'yearOfLoss']
Only in po

In [44]:
print(len(c), len(p))

84 87


In [45]:
# Experiment 1: what is the real maximum page size?
r = get("NfipClaims", **{"$top": 20000, "$filter": "state eq 'NC'",
                         "$select": "id", "$orderby": "id"})
print(len(r["NfipClaims"]))

20000


In [46]:
# Experiment 2: page through NC claims since 2019, 1,000 at a time
rows, skip = [], 0
while True:
    r = get("NfipClaims", **{"$top": 1000, "$skip": skip, "$select": "id",
                             "$filter": "state eq 'NC' and yearOfLoss ge 2019",
                             "$orderby": "id"})
    page = r["NfipClaims"]
    rows += page
    print(f"skip={skip}: got {len(page)} rows")
    if len(page) < 1000:
        break
    skip += 1000

ids = [x["id"] for x in rows]
print(len(ids), len(set(ids)))   # total rows, unique ids

skip=0: got 1000 rows
skip=1000: got 1000 rows
skip=2000: got 1000 rows
skip=3000: got 1000 rows
skip=4000: got 1000 rows
skip=5000: got 1000 rows
skip=6000: got 1000 rows
skip=7000: got 1000 rows
skip=8000: got 984 rows
8984 8984


In [47]:
c = claims.copy()
paid = c["amountPaidOnBuildingClaim"].fillna(0) + c["amountPaidOnContentsClaim"].fillna(0)

checks = {
    "rows": len(c),
    "zero total payment": int((paid == 0).sum()),
    "negative payment": int((paid < 0).sum()),
    "building paid > building coverage": int((c["amountPaidOnBuildingClaim"] > c["totalBuildingInsuranceCoverage"]).sum()),
    "missing ratedFloodZone": int(c["ratedFloodZone"].isna().sum()),
    "distinct zone spellings": c["ratedFloodZone"].nunique(),
    "missing floodEvent": int(c["floodEvent"].isna().sum()),
    "reportedCity placeholders": c["reportedCity"].value_counts().head(3).to_dict(),
    "dateOfLoss type": str(c["dateOfLoss"].dtype),
    "duplicate ids": int(c["id"].duplicated().sum()),
}
for k, v in checks.items():
    print(f"{k}: {v}")

rows: 8984
zero total payment: 1676
negative payment: 1
building paid > building coverage: 5
missing ratedFloodZone: 10
distinct zone spellings: 24
missing floodEvent: 3681
reportedCity placeholders: {'Currently Unavailable': 8984}
dateOfLoss type: str
duplicate ids: 0
